# 📡 Notebook 1: Real-Time Comment Streaming

When thousands of people watch a live video, they expect to see comments appear **instantly**. But how do you push new comments to millions of viewers at once?

This notebook walks you through the evolution from naive polling to efficient Server-Sent Events (SSE) with Redis Pub/Sub.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why polling doesn't work for real-time features
- How Server-Sent Events (SSE) work and when to use them vs WebSockets
- How Redis Pub/Sub broadcasts messages across servers
- How to build a complete real-time comment feed

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/fb-live-comments
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `live_comments`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
# === Connection Setup ===
# Connect to PostgreSQL and Redis, and verify both are reachable.

import psycopg2
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "live_comments",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
}

# --- Test PostgreSQL ---
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM comments")
    comment_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM users")
    user_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM live_videos")
    video_count = cur.fetchone()[0]
    cur.close()
    conn.close()
    print(f"✅ PostgreSQL connected — {comment_count} comments, {user_count} users, {video_count} live videos")
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")

# --- Test Redis ---
try:
    r = redis.Redis(**REDIS_CONFIG)
    r.ping()
    print("✅ Redis connected")
    r.close()
except Exception as e:
    print(f"❌ Redis connection failed: {e}")

## 🤔 The Problem: How Do Viewers See New Comments?

When a user posts a comment, it gets saved to the database. Easy enough.  
But how do the **other 10,000 viewers** see it?

The server can't just call everyone — HTTP only works when the **client** initiates a request.  
So we need a strategy to get new data from the server to the browser.

There are three main approaches:

```
+--------------+-------------------+------------------+-----------------+
| Approach     | How It Works      | Pros             | Cons            |
+--------------+-------------------+------------------+-----------------+
| Polling      | Client asks every | Simple to build  | Wastes network  |
|              | N seconds         |                  | & DB resources  |
+--------------+-------------------+------------------+-----------------+
| WebSockets   | Persistent two-   | True real-time,  | Complex, needs  |
|              | way connection    | bidirectional    | special infra   |
+--------------+-------------------+------------------+-----------------+
| SSE (Server- | Server pushes     | Simple, auto-    | One-way only    |
|  Sent Events)| events to client  | reconnect, HTTP  | (server→client) |
+--------------+-------------------+------------------+-----------------+
```

Let's explore each one and see why **SSE is the sweet spot** for live comments.

## 🔄 Approach 1: Polling (The Naive Way)

Polling is the simplest approach: the client keeps asking the server "got anything new?" every few seconds.

Think of it like refreshing your email inbox over and over — most of the time, there's nothing new,
but you keep checking anyway.

```
Client                          Server
  |--- GET /comments?since=100 --->|
  |<-- 200 OK (empty) -------------|   ← wasted request!
  |                                |
  |  (wait 2 seconds...)           |
  |                                |
  |--- GET /comments?since=100 --->|
  |<-- 200 OK (empty) -------------|   ← wasted request!
  |                                |
  |  (wait 2 seconds...)           |
  |                                |
  |--- GET /comments?since=100 --->|
  |<-- 200 OK [{id:101, ...}] -----|   ← finally, new data!
```

Let's simulate this and count how many queries are wasted.

In [ ]:
# === Polling Simulation ===
# We'll simulate a client polling the database for new comments.

import psycopg2
import time


def poll_new_comments(video_id: int, last_seen_id: int, limit: int = 20):
    """
    Simulate one poll: query the database for comments newer than last_seen_id.
    Returns a list of new comments (may be empty).
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute(
        "SELECT id, user_id, message, created_at FROM comments "
        "WHERE live_video_id = %s AND id > %s ORDER BY id ASC LIMIT %s",
        (video_id, last_seen_id, limit),
    )
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return [{"id": r[0], "user_id": r[1], "message": r[2], "created_at": str(r[3])} for r in rows]


# --- Simulate a polling loop ---
# We'll use a very high last_seen_id so most polls return empty (realistic scenario).

# First, find the max comment ID so we can start from a point with no new comments
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("SELECT MAX(id) FROM comments WHERE live_video_id = 1")
max_id = cur.fetchone()[0]
cur.close()
conn.close()

print(f"Starting poll from comment ID {max_id} (no new comments exist yet)\n")

total_queries = 0
queries_with_data = 0
poll_interval = 2  # seconds
num_polls = 3

for i in range(num_polls):
    start = time.time()
    new_comments = poll_new_comments(video_id=1, last_seen_id=max_id)
    elapsed_ms = (time.time() - start) * 1000
    total_queries += 1

    if new_comments:
        queries_with_data += 1
        print(f"  Poll {i+1}: {len(new_comments)} new comment(s) ({elapsed_ms:.1f}ms)")
    else:
        print(f"  Poll {i+1}: no new comments ({elapsed_ms:.1f}ms) ← wasted query!")

    if i < num_polls - 1:
        time.sleep(poll_interval)

print(f"\n📊 Polling Stats:")
print(f"   Total queries:       {total_queries}")
print(f"   Queries with data:   {queries_with_data}")
print(f"   Wasted queries:      {total_queries - queries_with_data}")
print(f"   Waste ratio:         {(total_queries - queries_with_data) / total_queries * 100:.0f}%")

### Why Polling Fails at Scale

The math is brutal. Imagine a moderately popular live stream:

- **10,000 viewers** watching a live video
- Each polls every **2 seconds**
- That's **5,000 requests/second** hitting your database
- In a 1-hour stream: **18 million queries**
- Most of those return **empty results** — pure waste!

```
Viewers    Poll Interval    Requests/sec    Queries/hour
1,000      2s               500             1,800,000
10,000     2s               5,000           18,000,000
100,000    2s               50,000          180,000,000  💀
```

Even with caching, this creates enormous unnecessary load.  
We need the server to **push** data to clients, not the other way around.

## 📡 Approach 2: Server-Sent Events (SSE)

SSE flips the model: instead of the client asking repeatedly, the server **pushes** data to the client
whenever something new happens.

### How SSE Works

1. The client opens a long-lived HTTP connection to the server
2. The server holds the connection open
3. Whenever there's new data, the server writes it to the connection
4. The client receives the data instantly — no polling needed!

### The SSE Protocol

SSE uses a simple text-based format. Each event looks like this:

```
event: comment
id: 12345
data: {"user": "Alice", "message": "Great stream!"}

```

- `event:` — the type of event (so clients can handle different kinds)
- `id:` — a unique ID (used for reconnection — more on this later)
- `data:` — the actual payload (JSON in our case)
- The blank line at the end marks the end of one event

### SSE vs WebSockets: When to Use Which?

Think of it this way:
- **WebSockets** = a **phone call** — both sides can talk at any time
- **SSE** = a **radio broadcast** — the server talks, clients listen

For live comments, the read/write ratio is **extremely imbalanced**:  
- 10,000 viewers **reading** comments (server → client)  
- Maybe 50 users **posting** comments per second (client → server via regular POST)  

SSE is perfect here because:
- It works over **standard HTTP** (no protocol upgrade like WebSockets)
- It has **built-in reconnection** with `Last-Event-ID`
- It's **simpler** to implement, deploy, and debug
- Regular HTTP POST handles the (rare) writes

## 🔴 Building a Live Comment System with Redis Pub/Sub

Now let's build it for real! Here's the architecture:

```
Commenter → POST /comments → Save to DB → PUBLISH to Redis
                                                  ↓
Viewers ← SSE stream ← Server ← SUBSCRIBE to Redis channel
```

### Why Redis Pub/Sub?

In production, you'll have **multiple servers** handling connections. When one server receives a
new comment, the others need to know about it too.

Redis Pub/Sub solves this:
- When a comment is posted, we **PUBLISH** it to a Redis channel (e.g., `live_video:1:comments`)
- All servers **SUBSCRIBE** to that channel
- Redis instantly delivers the message to every subscriber
- Each server then pushes it to its connected viewers via SSE

It's **fire-and-forget**: Redis doesn't store messages. That's fine because we persist
comments to PostgreSQL. Redis is just the real-time broadcast layer.

In [ ]:
# === Step 1: Understanding Redis Pub/Sub ===
# Before building the full server, let's see Pub/Sub in action.

import redis
import json
import threading
import time

CHANNEL = "live_video:1:comments"
received_messages = []


def subscriber_thread():
    """Runs in a background thread, listening for messages on the channel."""
    r = redis.Redis(**REDIS_CONFIG)
    pubsub = r.pubsub()
    pubsub.subscribe(CHANNEL)
    for message in pubsub.listen():
        if message["type"] == "message":
            data = json.loads(message["data"].decode())
            received_messages.append(data)
            print(f"   📥 Subscriber received: {data['message']}")
            if len(received_messages) >= 2:
                break
    pubsub.unsubscribe()
    r.close()


# Start a subscriber in the background
listener = threading.Thread(target=subscriber_thread, daemon=True)
listener.start()
time.sleep(0.5)  # Give the subscriber time to connect

# Publish two messages
r = redis.Redis(**REDIS_CONFIG)

msg1 = {"user": "Alice", "message": "Hello from Pub/Sub! 👋"}
msg2 = {"user": "Bob", "message": "I heard you instantly! ⚡"}

print(f"📤 Publishing: {msg1['message']}")
r.publish(CHANNEL, json.dumps(msg1))
time.sleep(0.5)

print(f"📤 Publishing: {msg2['message']}")
r.publish(CHANNEL, json.dumps(msg2))

listener.join(timeout=5)
r.close()

print(f"\n✅ Published 2 messages, received {len(received_messages)} — instant delivery!")

In [ ]:
# === Step 2: Building the FastAPI Server ===
# This cell writes a complete FastAPI app with:
#   POST /comments/{video_id}  — saves to DB and publishes to Redis
#   GET  /stream/{video_id}    — SSE endpoint that subscribes to Redis

import os

server_code = '''import asyncio
import json
from fastapi import FastAPI
from pydantic import BaseModel
from sse_starlette.sse import EventSourceResponse
import psycopg2
import redis

app = FastAPI(title="FB Live Comments")

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "live_comments",
    "user": "demo",
    "password": "demo",
}


class CommentCreate(BaseModel):
    user_id: int
    message: str


@app.post("/comments/{video_id}")
def post_comment(video_id: int, comment: CommentCreate):
    """Save a comment to PostgreSQL and broadcast it via Redis Pub/Sub."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO comments (live_video_id, user_id, message) "
        "VALUES (%s, %s, %s) RETURNING id, created_at",
        (video_id, comment.user_id, comment.message),
    )
    comment_id, created_at = cur.fetchone()
    conn.commit()
    cur.close()
    conn.close()

    # Publish to Redis so all SSE servers can broadcast it
    r = redis.Redis(host="localhost", port=6379)
    event_data = json.dumps({
        "id": comment_id,
        "video_id": video_id,
        "user_id": comment.user_id,
        "message": comment.message,
        "created_at": str(created_at),
    })
    r.publish(f"live_video:{video_id}:comments", event_data)
    r.close()
    return {"id": comment_id, "status": "published"}


@app.get("/stream/{video_id}")
async def stream_comments(video_id: int):
    """SSE endpoint: streams new comments in real-time via Redis Pub/Sub."""
    async def event_generator():
        r = redis.Redis(host="localhost", port=6379)
        pubsub = r.pubsub()
        pubsub.subscribe(f"live_video:{video_id}:comments")
        try:
            while True:
                message = pubsub.get_message(timeout=1.0)
                if message and message["type"] == "message":
                    data = message["data"].decode()
                    comment = json.loads(data)
                    yield {
                        "event": "comment",
                        "id": str(comment["id"]),
                        "data": data,
                    }
                await asyncio.sleep(0.1)
        finally:
            pubsub.unsubscribe()
            r.close()

    return EventSourceResponse(event_generator())
'''

# Write the server to a file inside the project directory
server_dir = os.path.join(os.path.dirname(os.getcwd()), "_server_tmp")
os.makedirs(server_dir, exist_ok=True)
server_path = os.path.join(server_dir, "fb_live_comments_server.py")

with open(server_path, "w") as f:
    f.write(server_code)

print(f"✅ Server code written to {server_path}")
print()
print("Endpoints:")
print("  POST /comments/{{video_id}}  — post a comment (saves to DB + publishes to Redis)")
print("  GET  /stream/{{video_id}}    — SSE stream (subscribes to Redis Pub/Sub)")

In [ ]:
# === Start the FastAPI Server ===

import subprocess
import time

server_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "fb_live_comments_server:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=server_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(3)  # Give the server time to start up

# Verify the server is running
if server_process.poll() is None:
    print(f"🚀 Server started (PID: {server_process.pid})")
    print(f"   POST http://localhost:8000/comments/{{video_id}}")
    print(f"   SSE  http://localhost:8000/stream/{{video_id}}")
else:
    print("❌ Server failed to start!")
    print(server_process.stderr.read().decode())

### Step 3: See It in Action!

Now let's test the full flow:

1. **Open an SSE connection** in a background thread (simulating a viewer)
2. **Post 3 comments** via the REST API (simulating commenters)
3. **Watch the comments arrive** in real-time through the SSE stream

The key thing to notice: the viewer receives each comment **instantly** after it's posted,  
without ever asking the server "do you have anything new?"

In [ ]:
# === Demo: Full Real-Time Flow ===
# Open an SSE listener, post comments, watch them arrive instantly.

import httpx
import threading
import time
import json

received_comments = []


def listen_for_comments(video_id):
    """Listen to the SSE stream in a background thread (simulates a viewer)."""
    with httpx.Client(timeout=None) as client:
        with client.stream("GET", f"http://localhost:8000/stream/{video_id}") as response:
            for line in response.iter_lines():
                if line.startswith("data:"):
                    comment = json.loads(line[5:].strip())
                    received_comments.append(comment)
                    if len(received_comments) >= 3:
                        return


# Start listening in background (like a viewer opening the live stream page)
listener = threading.Thread(target=listen_for_comments, args=(1,), daemon=True)
listener.start()
time.sleep(1)  # Let the SSE connection establish

# Post 3 comments (like users typing in the chat)
test_comments = [
    (5, "Hello from the notebook! 👋"),
    (12, "This SSE stuff is cool!"),
    (30, "Real-time comments work! 🎉"),
]

for user_id, message in test_comments:
    response = httpx.post(
        "http://localhost:8000/comments/1",
        json={"user_id": user_id, "message": message},
    )
    print(f"📤 Posted: {message} → ID: {response.json()['id']}")
    time.sleep(0.5)

# Wait for the listener to receive all comments
time.sleep(2)
listener.join(timeout=5)

print(f"\n📥 Received {len(received_comments)} comments via SSE:")
for c in received_comments:
    print(f"   💬 [User {c['user_id']}]: {c['message']}")

## 🔌 Handling Disconnections

In the real world, connections drop all the time — your phone walks through a tunnel,
your Wi-Fi hiccups, or you switch from cellular to Wi-Fi.

SSE has a **built-in reconnection mechanism** using the `Last-Event-ID` header.

### How It Works

```
1. Viewer is happily receiving comments...
   ← event: comment, id: 501
   ← event: comment, id: 502

2. 💥 Connection drops! (tunnel, Wi-Fi switch, etc.)
   (Comments 503, 504, 505 are posted while viewer is offline)

3. Browser automatically reconnects with:
   → GET /stream/1
   → Last-Event-ID: 502    ← "I last saw comment 502"

4. Server sees Last-Event-ID, queries DB:
   SELECT * FROM comments WHERE id > 502 ORDER BY id ASC

5. Server sends missed comments first, then resumes live stream:
   ← event: comment, id: 503  (catch-up)
   ← event: comment, id: 504  (catch-up)
   ← event: comment, id: 505  (catch-up)
   ← (now back to live stream from Redis Pub/Sub)
```

This is why we **persist comments to PostgreSQL** even though Redis Pub/Sub handles
real-time delivery. The database is our **catch-up safety net**.

Remember: Redis Pub/Sub is fire-and-forget — if nobody is listening when a message is
published, it's gone. The database ensures we never lose a comment.

In [ ]:
# === Catch-Up Logic ===
# When a client reconnects, it sends the last comment ID it saw.
# The server queries PostgreSQL for any comments it missed.

import psycopg2


def catch_up_comments(video_id: int, last_seen_id: int, limit: int = 50):
    """
    Fetch comments that the client missed while disconnected.
    This is the query the server would run when it sees a Last-Event-ID header.
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute(
        "SELECT c.id, c.user_id, u.display_name, c.message, c.created_at "
        "FROM comments c JOIN users u ON c.user_id = u.id "
        "WHERE c.live_video_id = %s AND c.id > %s "
        "ORDER BY c.id ASC LIMIT %s",
        (video_id, last_seen_id, limit),
    )
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return [
        {"id": r[0], "user_id": r[1], "display_name": r[2],
         "message": r[3], "created_at": str(r[4])}
        for r in rows
    ]


# Demo: Simulate reconnection after missing some comments.
# Pretend the client last saw comment ID 10 and disconnected.
last_seen = 10
missed = catch_up_comments(video_id=1, last_seen_id=last_seen, limit=10)

print(f"🔌 Client reconnected with Last-Event-ID: {last_seen}")
print(f"📦 Catching up — found {len(missed)} missed comments:\n")

for c in missed:
    print(f"   💬 #{c['id']} [{c['display_name']}]: {c['message'][:60]}")

if missed:
    print(f"\n✅ Client is now caught up to comment #{missed[-1]['id']}")
    print("   Server would now switch to live Redis Pub/Sub stream.")

## 📊 Polling vs SSE: A Side-by-Side Comparison

Let's put numbers on the difference between polling and SSE.

| Metric | Polling (2s interval) | SSE |
|--------|----------------------|-----|
| Connection model | Open → close → open → close | Open once, stay connected |
| Latency | 0–2 seconds (avg 1s) | Near-instant (< 100ms) |
| Wasted requests | ~90% (most polls return empty) | 0% (only sends when data exists) |
| Server resources | High (handle thousands of short requests) | Low (hold open connections) |
| Reconnection | Manual (client must track state) | Automatic (Last-Event-ID) |
| Protocol | HTTP request/response | HTTP streaming |

In [ ]:
# === Quantitative Comparison: Polling vs SSE ===
# Let's calculate the actual load for a 1-hour live stream.

viewers = 10_000
stream_duration_hours = 1
stream_duration_seconds = stream_duration_hours * 3600
poll_interval_seconds = 2
comments_per_minute = 30  # Average comments posted per minute

# --- Polling ---
polls_per_viewer = stream_duration_seconds / poll_interval_seconds
total_polling_requests = viewers * polls_per_viewer
total_comments = comments_per_minute * 60 * stream_duration_hours
# Each poll might hit a new comment ~once every few polls
useful_polls_per_viewer = total_comments / viewers  # Each viewer needs to see each comment once
wasted_polls = total_polling_requests - (viewers * useful_polls_per_viewer)

# --- SSE ---
sse_connections = viewers  # One long-lived connection per viewer
sse_messages_sent = total_comments * viewers  # Each comment is pushed to each viewer
sse_requests = viewers  # Just the initial connection

print("📊 Load Comparison: 10,000 Viewers, 1-Hour Live Stream")
print("=" * 60)
print()
print(f"{'Metric':<30} {'Polling':<20} {'SSE':<20}")
print(f"{'-'*30} {'-'*20} {'-'*20}")
print(f"{'HTTP requests':<30} {total_polling_requests:>15,.0f} {sse_requests:>15,.0f}")
print(f"{'DB queries':<30} {total_polling_requests:>15,.0f} {total_comments:>15,.0f}")
print(f"{'Messages delivered':<30} {total_comments * viewers:>15,.0f} {total_comments * viewers:>15,.0f}")
print(f"{'Wasted requests':<30} {wasted_polls:>15,.0f} {'0':>15}")
print(f"{'Avg latency':<30} {'~1 second':>15} {'< 100ms':>15}")
print()
print(f"💡 SSE eliminates {total_polling_requests - sse_requests:,.0f} unnecessary HTTP requests!")
print(f"   That's {(total_polling_requests - sse_requests) / total_polling_requests * 100:.1f}% less network overhead.")

In [ ]:
# === Cleanup ===
# Stop the FastAPI server and remove temp files.

import os
import shutil

server_process.terminate()
server_process.wait()
print("🧹 Server stopped")

# Clean up the temp server directory
if os.path.exists(server_dir):
    shutil.rmtree(server_dir)
    print("🧹 Temp files cleaned up")

print("\n✅ All done! Move on to the next notebook.")

## 📚 Summary

### Key Takeaways

1. **Polling wastes resources** — most requests return empty when there are no new comments
2. **SSE is ideal for live feeds** — one-way push from server, built-in reconnection, works over standard HTTP
3. **Redis Pub/Sub** enables broadcasting — when a comment is posted, all servers hear about it instantly
4. **Disconnection handling** is critical — use Last-Event-ID and catch-up queries to fill gaps

### How This Maps to the System Design Interview

| Concept | What to Say |
|---------|-------------|
| Real-time delivery | "I'd use SSE over WebSockets because the read/write ratio is highly imbalanced" |
| Pub/Sub | "Redis Pub/Sub broadcasts comments to all servers; fire-and-forget is fine since we persist to DB" |
| Reconnection | "SSE's Last-Event-ID header handles reconnection; server replays missed comments from DB" |

### Next Up

In **Notebook 2**, we'll tackle **comment ordering and pagination** — how to efficiently load historical comments with cursor-based pagination.